<a href="https://colab.research.google.com/github/mdaminu2002-sketch/bank_fraud/blob/main/waste_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("joebeachcapital/realwaste")

print("Path to dataset files:", path)

100%|██████████| 657M/657M [00:09<00:00, 73.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/joebeachcapital/realwaste/versions/1


In [3]:
#from posix import listdir
print(os.listdir(path))

['realwaste-main']


In [4]:
for item in os.listdir(path):
  full_path = os.path.join(path, item)
  if os.path.isfile(full_path):
    content = os.listdir(full_path)
    print(f"{item} / {len(content)} items & as for first few {content[:3]}")
  else:
    print(f"{item} / is a folder")
#

realwaste-main / is a folder


In [5]:
for item in os.listdir(path):
  print(item)

realwaste-main


In [6]:
inner_path = os.path.join(path, "realwaste-main")

for item in os.listdir(inner_path):
  full_path = os.path.join(inner_path, item)
  if os.path.isdir(full_path):
    content = os.listdir(full_path)
    print(f"{item} / {len(content)} items & as for first few {content[:3]}")
  else:
    print(f"{item} / folder")

RealWaste / 9 items & as for first few ['Miscellaneous Trash', 'Metal', 'Plastic']
README.md / folder


In [12]:
data_path = os.path.join(inner_path, "RealWaste")

for item in os.listdir(data_path):
  full_path = os.path.join(data_path, item)
  if os.path.isdir(full_path):
    count = len(os.listdir(full_path))
    print(f"{item} / {count} images")

Miscellaneous Trash / 495 images
Metal / 790 images
Plastic / 921 images
Glass / 420 images
Textile Trash / 318 images
Cardboard / 461 images
Food Organics / 411 images
Paper / 500 images
Vegetation / 436 images


In [21]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
import torch
import torchvision.models as models


In [14]:
|train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [20]:
full_dataset = ImageFolder(root=data_path, transform=train_transform)
print(full_dataset.classes)
print(len(full_dataset))



train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(42))

val_dataset.dataset.transform = val_test_transform
test_dataset.dataset.transform = val_test_transform

['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']
4752


In [22]:
model = models.resnet18(pretrained = True)

for param in model.parameters():
  param.requires_grad = False

num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, 9)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 241MB/s]
